In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
with open("shakespeare.txt","r", encoding="utf-8") as f:
    data=f.read().lower()

print("Dataset Loaded")
print(data[:300])

Dataset Loaded
the sonnets

by william shakespeare

from fairest creatures we desire increase,
that thereby beauty's rose might never die,
but as the riper should by time decease,
his tender heir might bear his memory:
but thou contracted to thine own bright eyes,
feed'st thy light's flame with self-substantial fu


#### Preprocessing

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

total_words = len(tokenizer.word_index) +1

input_sequences = []
for line in data.split("/n"):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]

input_sequences.append(n_gram_seq)

In [4]:
max_seq_len = max(len(x) for x in  input_sequences)

input_sequences = np.array(
    pad_sequences(input_sequences,
    maxlen=max_seq_len,padding='pre')
)

#### Split into X and y

In [5]:
X = input_sequences[:,:-1]
y = input_sequences[:,-1]

y = tf.keras.utils.to_categorical(y,num_classes= total_words)

print("Preprocessing Done")
print("X shape:", X.shape)
print("y shape:", y.shape)

Preprocessing Done
X shape: (1, 17674)
y shape: (1, 3201)


#### model Building

In [6]:
model = Sequential()

model.add(Embedding(total_words, 100,
input_length=max_seq_len-1))
model.add(LSTM(100))
model.add(Dense(total_words,
          activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

C:\Users\Dell\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(X,y, epochs=5, verbose=1)

In [ ]:
def generate_text(seed_text, next_words=20):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        tokrn_list = pad_sequences([token_list]),maxlen=mac_seq_len-1,padding='pre')
        predicted = np.argmax(model.predict(token_list),axis=-1)

        for word, index in tokenizer.wor_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

In [ ]:
print(generate_text("shall i compare thee",10))
print(generate_text("love is not", 20))
print(generate_text("my heart", 20))